# AI-Powered Conditional Image Colorization
### User-Guided Region Colorization (Sky, Grass, Water, Buildings, Roads, Sidewalks, Mountains, Indoor elements)

This notebook implements a complete conditional image colorization pipeline. The system uses:
1. **Semantic Segmentation**: Hugging Face's lightweight `nvidia/segformer-b0-finetuned-ade20k` model to extract region masks.
2. **Text Prompt Parser**: A keyword-matching engine to parse natural language color requests.
3. **U-Net Color Propagator**: A PyTorch U-Net model trained on-the-fly to propagate sparse user color hints (from dropdowns, color pickers, or text prompts) across segmented region boundaries.
4. **Gradio GUI**: A side-by-side dashboard showing the Original, Segmentation, and Colorized images.


## 1. Dependency Installation & Workspace Initialization
This cell installs all required libraries (including `gradio` and `transformers`) and creates outputs folders under `/content/`.


In [1]:
# Install required libraries in the Colab environment
!pip install -q gradio transformers torch torchvision scikit-image pandas numpy matplotlib seaborn opencv-python-headless

# Create required directories under /content/
import os
directories = [
    "dataset",
    "outputs/original",
    "outputs/colorized",
    "outputs/comparisons",
    "outputs/metrics",
    "outputs/visualizations"
]
for d in directories:
    os.makedirs(d, exist_ok=True)

print("Environment setup completed. Directories created under /content/.")


Environment setup completed. Directories created under /content/.


## 2. Sample Image Dataset Download
We fetch a diverse set of 11 public domain images representing landscapes, urban scenes, and indoor environments.
If any download fails, a numpy-based synthetic image is generated as a fallback to ensure the training loop runs successfully.


In [2]:
import os
import requests
import numpy as np
import cv2
from PIL import Image

image_sources = [
    {"url": "https://upload.wikimedia.org/wikipedia/commons/e/eb/Mount_Fuji_from_Kawaguchiko_2003-11-05.jpg", "name": "fuji.jpg"},
    {"url": "https://upload.wikimedia.org/wikipedia/commons/3/35/Val_di_Funes_in_July_2016.jpg", "name": "funes.jpg"},
    {"url": "https://upload.wikimedia.org/wikipedia/commons/c/c5/Moraine_Lake_17092005.jpg", "name": "moraine.jpg"},
    {"url": "https://upload.wikimedia.org/wikipedia/commons/4/47/New_york_times_square-terabass.jpg", "name": "times_square.jpg"},
    {"url": "https://upload.wikimedia.org/wikipedia/commons/a/a8/Sydney_Harbour_Bridge_from_the_air.jpg", "name": "sydney.jpg"},
    {"url": "https://upload.wikimedia.org/wikipedia/commons/2/25/Row_houses_in_San_Francisco.jpg", "name": "sf_houses.jpg"},
    {"url": "https://upload.wikimedia.org/wikipedia/commons/e/eb/1950s_kitchen.jpg", "name": "kitchen.jpg"},
    {"url": "https://upload.wikimedia.org/wikipedia/commons/8/84/Lobby_at_the_Taj_Mahal_Palace_Hotel.jpg", "name": "lobby.jpg"},
    {"url": "https://upload.wikimedia.org/wikipedia/commons/5/5f/Living_Room_in_Sweden.jpg", "name": "living_room.jpg"},
    {"url": "https://upload.wikimedia.org/wikipedia/commons/b/b2/Red_Tulips_in_Keukenhof.jpg", "name": "tulips.jpg"},
    {"url": "https://upload.wikimedia.org/wikipedia/commons/9/91/White_Clouds_Blue_Sky.jpg", "name": "sky_clouds.jpg"}
]

def generate_synthetic_image(path, name):
    # Generates a colorful synthetic image
    img = np.zeros((512, 512, 3), dtype=np.uint8)
    if any(k in name for k in ["fuji", "funes", "moraine", "sky", "tulips"]):
        # Landscape: blue sky top, green grass bottom
        img[:256, :] = [235, 206, 135] # sky (BGR)
        img[256:, :] = [50, 205, 50] # grass
        pts = np.array([[100, 256], [200, 100], [300, 256]], np.int32)
        cv2.fillPoly(img, [pts], [19, 69, 139]) # brown mountain
    elif any(k in name for k in ["kitchen", "lobby", "living"]):
        # Indoor: gray walls, brown floor
        img[:350, :] = [220, 220, 220] # gray wall
        img[350:, :] = [30, 105, 210] # brown floor
        cv2.rectangle(img, (150, 300), (350, 420), (50, 50, 200), -1) # sofa
    else:
        # Urban: buildings and gray road
        img[:300, :] = [180, 180, 180] # buildings
        img[300:, :] = [128, 128, 128] # road
        cv2.line(img, (256, 300), (256, 512), (0, 255, 255), 5)

    cv2.imwrite(path, img)
    print(f"Generated synthetic image for {name}")

session = requests.Session()
adapter = requests.adapters.HTTPAdapter(max_retries=3)
session.mount("https://", adapter)

print("Starting dataset acquisition...")
for item in image_sources:
    save_path = os.path.join("dataset", item["name"])
    success = False
    try:
        r = session.get(item["url"], timeout=10)
        if r.status_code == 200:
            with open(save_path, "wb") as f:
                f.write(r.content)
            with Image.open(save_path) as img:
                img.verify()
            print(f"Downloaded: {item['name']}")
            success = True
    except Exception as e:
        print(f"Failed to download {item['name']}: {e}")

    if not success:
        generate_synthetic_image(save_path, item["name"])

print("Dataset preparation completed.")


Starting dataset acquisition...
Generated synthetic image for fuji.jpg
Generated synthetic image for funes.jpg
Generated synthetic image for moraine.jpg
Generated synthetic image for times_square.jpg
Generated synthetic image for sydney.jpg
Generated synthetic image for sf_houses.jpg
Generated synthetic image for kitchen.jpg
Generated synthetic image for lobby.jpg
Generated synthetic image for living_room.jpg
Generated synthetic image for tulips.jpg
Generated synthetic image for sky_clouds.jpg
Dataset preparation completed.


## 3. SegFormer Segmentation Module
We load the pre-trained `nvidia/segformer-b0-finetuned-ade20k` model from Hugging Face to detect sky, grass, vegetation, water, buildings, roads, sidewalks, mountains, and indoor elements. We also set up custom color mapping to visualize segmentation maps.


In [5]:
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import torch
import torch.nn as nn
from PIL import Image
import numpy as np

# Load model
processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")
segmentation_model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")
segmentation_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
segmentation_model = segmentation_model.to(device)

# Mappings for ADE20K classes to user-friendly categories
class_mapping = {
    "sky": [2],
    "grass": [9],
    "vegetation": [4, 17, 72, 96], # tree, plant, shrub, canopy
    "water": [26, 109, 128], # water, river, lake
    "building": [1, 25, 48], # wall, house, skyscraper
    "road": [6, 114], # road, path
    "sidewalk": [11], # sidewalk
    "mountain": [21, 68], # mountain, hill
    "cloud": [40], # cloud
    "floor": [3], # floor
    "ceiling": [5] # ceiling
}

# Unique colors for each mapped class (RGB)
label_colors = {
    "sky": [135, 206, 235],       # Sky blue
    "grass": [34, 139, 34],        # Forest green
    "vegetation": [0, 100, 0],     # Dark green
    "water": [30, 144, 255],       # Dodger blue
    "building": [165, 42, 42],     # Brown/Red brick
    "road": [128, 128, 128],       # Gray road
    "sidewalk": [211, 211, 211],   # Light gray sidewalk
    "mountain": [105, 105, 105],   # Dim gray mountain
    "cloud": [240, 248, 255],     # Alice blue
    "floor": [244, 164, 96],      # Sandy brown
    "ceiling": [255, 250, 250],    # Snow white
    "other": [0, 0, 0]             # Black for background/unmapped
}

def segment_image(img_pil):
    inputs = processor(images=img_pil, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = segmentation_model(**inputs)

    # Resize logits to original image size
    logits = outputs.logits
    upsampled_logits = nn.functional.interpolate(
        logits,
        size=img_pil.size[::-1], # (H, W)
        mode="bilinear",
        align_corners=False
    )

    labels = upsampled_logits.argmax(dim=1)[0].cpu().numpy()
    return labels

def get_segmentation_colormap(labels):
    H, W = labels.shape
    color_map = np.zeros((H, W, 3), dtype=np.uint8)

    for class_name, ids in class_mapping.items():
        mask = np.isin(labels, ids)
        color_map[mask] = label_colors[class_name]

    mapped_mask = np.zeros((H, W), dtype=bool)
    for ids in class_mapping.values():
        mapped_mask |= np.isin(labels, ids)
    color_map[~mapped_mask] = label_colors["other"]

    return color_map

print("SegFormer model and color maps successfully initialized.")


config.json:   0%|          | 0.00/6.88k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/15.0M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

SegFormer model and color maps successfully initialized.


## 4. Conditional Colorizer Model Definition (U-Net) & Training
Here, we define a lightweight 4-channel input PyTorch U-Net model and train it on-the-fly using L1 loss.
Hints are simulated dynamically by selecting ground truth colors inside segment masks.


In [6]:
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import random

# U-Net Architecture
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class SimpleUNet(nn.Module):
    def __init__(self, in_channels=4, out_channels=2):
        super().__init__()
        # Encoder
        self.inc = DoubleConv(in_channels, 32)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(32, 64))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))

        # Decoder
        self.up1 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv_up1 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv_up2 = DoubleConv(128, 64)

        self.up3 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.conv_up3 = DoubleConv(64, 32)

        self.outc = nn.Conv2d(32, out_channels, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)

        x = self.up1(x4)
        x = torch.cat([x, x3], dim=1)
        x = self.conv_up1(x)

        x = self.up2(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv_up2(x)

        x = self.up3(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv_up3(x)

        return self.outc(x)

# Custom Dataset with Hint Simulation
class ColorizationDataset(Dataset):
    def __init__(self, image_dir, image_files):
        self.image_dir = image_dir
        self.image_files = image_files
        self.resize = transforms.Resize((256, 256))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        path = os.path.join(self.image_dir, self.image_files[idx])
        img = Image.open(path).convert("RGB")
        img = self.resize(img)
        img_np = np.array(img)

        # Convert to LAB space using OpenCV
        img_lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB).astype(np.float32)
        L = img_lab[:, :, 0:1] - 50.0   # Zero-centered L
        ab = img_lab[:, :, 1:3] - 128.0 # Zero-centered ab

        L_tensor = torch.from_numpy(L).permute(2, 0, 1)
        ab_tensor = torch.from_numpy(ab).permute(2, 0, 1)

        labels = segment_image(img)

        hint_ab = torch.zeros((2, 256, 256), dtype=torch.float32)
        hint_mask = torch.zeros((1, 256, 256), dtype=torch.float32)

        available_classes = []
        for class_name, ids in class_mapping.items():
            mask = np.isin(labels, ids)
            if mask.sum() > 800:
                available_classes.append(class_name)

        if len(available_classes) > 0:
            num_hints = random.randint(1, min(2, len(available_classes)))
            selected_classes = random.sample(available_classes, num_hints)
            for cls in selected_classes:
                mask = np.isin(labels, class_mapping[cls])
                mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()

                # Overlay
                hint_mask = torch.max(hint_mask, mask_tensor)
                hint_ab = hint_ab + ab_tensor * mask_tensor

        return L_tensor, ab_tensor, hint_ab, hint_mask

# Dataset initialization & on-the-fly training
image_files = [f for f in os.listdir("dataset") if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
dataset = ColorizationDataset("dataset", image_files)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

unet_model = SimpleUNet().to(device)
optimizer = optim.Adam(unet_model.parameters(), lr=1e-3)
criterion = nn.L1Loss()

epochs = 15
print("Starting lightweight on-the-fly model training...")
unet_model.train()

for epoch in range(epochs):
    epoch_loss = 0.0
    for L_batch, ab_batch, hint_ab_batch, hint_mask_batch in loader:
        L_batch = L_batch.to(device)
        ab_batch = ab_batch.to(device)
        hint_ab_batch = hint_ab_batch.to(device)
        hint_mask_batch = hint_mask_batch.to(device)

        inputs = torch.cat([L_batch, hint_ab_batch, hint_mask_batch], dim=1)

        optimizer.zero_grad()
        outputs = unet_model(inputs)
        loss = criterion(outputs, ab_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * L_batch.size(0)

    avg_loss = epoch_loss / len(dataset)
    print(f"Epoch [{epoch+1}/{epochs}], L1 Loss: {avg_loss:.4f}")

# Save the trained weights
torch.save(unet_model.state_dict(), "outputs/metrics/conditional_colorizer.pth")
print("Model trained and weights saved to outputs/metrics/conditional_colorizer.pth.")


Starting lightweight on-the-fly model training...
Epoch [1/15], L1 Loss: 23.6023
Epoch [2/15], L1 Loss: 23.4071
Epoch [3/15], L1 Loss: 23.3126
Epoch [4/15], L1 Loss: 23.2718
Epoch [5/15], L1 Loss: 23.1180
Epoch [6/15], L1 Loss: 23.2378
Epoch [7/15], L1 Loss: 23.3198
Epoch [8/15], L1 Loss: 23.0868
Epoch [9/15], L1 Loss: 23.0876
Epoch [10/15], L1 Loss: 23.0902
Epoch [11/15], L1 Loss: 22.9794
Epoch [12/15], L1 Loss: 23.0915
Epoch [13/15], L1 Loss: 22.9544
Epoch [14/15], L1 Loss: 22.9424
Epoch [15/15], L1 Loss: 22.8741
Model trained and weights saved to outputs/metrics/conditional_colorizer.pth.


## 5. Text Prompt Parser & Rule Engine
This section implements a simple keyword-based natural language parser to extract color criteria from a free-text prompt and convert color names/hex codes into CIE L*a*b* space values.


In [7]:
import re

def parse_text_prompt(prompt_text):
    text = prompt_text.lower()

    region_synonyms = {
        "sky": ["sky", "clouds", "cloud"],
        "grass": ["grass", "lawn"],
        "vegetation": ["vegetation", "tree", "trees", "plant", "plants", "bush", "bushes", "forest", "foliage", "leaves"],
        "road": ["road", "roads", "highway", "street", "streets", "asphalt"],
        "sidewalk": ["sidewalk", "sidewalks", "path", "walkway"],
        "water": ["water", "sea", "lake", "ocean", "river", "pond"],
        "building": ["building", "buildings", "wall", "house", "structure", "structures"],
        "mountain": ["mountain", "mountains", "hill", "hills", "rock", "rocks"],
        "floor": ["floor", "carpet", "ground"],
        "ceiling": ["ceiling", "roof"]
    }

    color_palette = {
        "deep blue": [10, 30, 150],
        "light blue": [173, 216, 230],
        "blue": [30, 144, 255],
        "bright green": [50, 205, 50],
        "dark green": [34, 139, 34],
        "lush green": [46, 139, 87],
        "green": [0, 128, 0],
        "dark gray": [64, 64, 64],
        "light gray": [211, 211, 211],
        "gray": [128, 128, 128],
        "grey": [128, 128, 128],
        "dark grey": [64, 64, 64],
        "light grey": [211, 211, 211],
        "red": [220, 20, 60],
        "brick red": [178, 34, 34],
        "orange": [255, 140, 0],
        "sunset orange": [255, 69, 0],
        "yellow": [255, 215, 0],
        "gold": [255, 215, 0],
        "brown": [139, 69, 19],
        "earthy brown": [139, 69, 19],
        "white": [245, 245, 245],
        "black": [30, 30, 30]
    }

    rules = {}
    clauses = re.split(r'[,;.]', text)
    for clause in clauses:
        matched_region = None
        for region_key, synonyms in region_synonyms.items():
            for syn in synonyms:
                if re.search(r'' + syn + r'', clause):
                    matched_region = region_key
                    break
            if matched_region:
                break

        if matched_region:
            matched_color = None
            for color_name, rgb in sorted(color_palette.items(), key=lambda x: -len(x[0])):
                if color_name in clause:
                    matched_color = rgb
                    break
            if matched_color:
                rules[matched_region] = matched_color

    return rules

def hex_to_rgb(hex_str):
    hex_str = hex_str.lstrip('#')
    return [int(hex_str[i:i+2], 16) for i in (0, 2, 4)]

print("Natural Language prompt keyword matcher initialized.")


Natural Language prompt keyword matcher initialized.


## 6. Integrated Inference Pipeline & Interactive Gradio GUI
This section combines the components into an integrated inference pipeline. It also sets up a **Gradio GUI** that displays the Original Image, Segmented Classes, and Colorized Output side-by-side.
It lists the detected editable regions immediately upon image upload.


In [11]:
import gradio as gr

# Pipeline Function
def colorize_image_pipeline(img_pil, text_prompt="", dropdown_rules=None, blend_weight=0.8):
    W_orig, H_orig = img_pil.size
    labels = segment_image(img_pil)

    # Compile rules
    combined_rules = {}
    if text_prompt.strip():
        combined_rules.update(parse_text_prompt(text_prompt))

    if dropdown_rules:
        for region, rgb_color in dropdown_rules.items():
            if rgb_color is not None:
                combined_rules[region] = rgb_color

    # Prepare original image in LAB
    img_np = np.array(img_pil.convert("RGB"))
    img_lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB).astype(np.float32)
    L_orig = img_lab[:, :, 0:1]

    # Prepare resized image for U-Net
    img_resized_np = np.array(img_pil.convert("RGB").resize((256, 256)))
    img_resized_lab = cv2.cvtColor(img_resized_np, cv2.COLOR_RGB2LAB).astype(np.float32)
    L_resized = img_resized_lab[:, :, 0:1] - 50.0

    L_tensor = torch.from_numpy(L_resized).permute(2, 0, 1).unsqueeze(0).to(device)
    hint_ab = torch.zeros((1, 2, 256, 256), dtype=torch.float32).to(device)
    hint_mask = torch.zeros((1, 1, 256, 256), dtype=torch.float32).to(device)

    for region, rgb in combined_rules.items():
        if region in class_mapping:
            ids = class_mapping[region]
            mask_orig = np.isin(labels, ids)
            if mask_orig.sum() == 0:
                continue

            # Resize mask to 256x256 for U-Net
            mask_256 = cv2.resize(mask_orig.astype(np.uint8), (256, 256), interpolation=cv2.INTER_NEAREST)
            mask_tensor = torch.from_numpy(mask_256).unsqueeze(0).unsqueeze(0).float().to(device)

            color_img = np.zeros((1, 1, 3), dtype=np.uint8)
            color_img[0, 0] = rgb
            color_lab = cv2.cvtColor(color_img, cv2.COLOR_RGB2LAB).astype(np.float32)
            target_ab = color_lab[0, 0, 1:3] - 128.0

            target_ab_tensor = torch.tensor(target_ab, dtype=torch.float32).view(1, 2, 1, 1).to(device)
            hint_ab = hint_ab + target_ab_tensor * mask_tensor
            hint_mask = torch.max(hint_mask, mask_tensor)

    # U-Net Inference
    unet_model.eval()
    inputs = torch.cat([L_tensor, hint_ab, hint_mask], dim=1)
    with torch.no_grad():
        ab_pred = unet_model(inputs)

    ab_pred_np = ab_pred.squeeze(0).permute(1, 2, 0).cpu().numpy()
    ab_pred_shifted = ab_pred_np + 128.0

    # Resize predicted ab back to original image size
    ab_orig_pred = cv2.resize(ab_pred_shifted, (W_orig, H_orig), interpolation=cv2.INTER_CUBIC)

    # Apply user color rules at high-resolution with blending weight
    final_ab = ab_orig_pred.copy()
    for region, rgb in combined_rules.items():
        if region in class_mapping:
            ids = class_mapping[region]
            mask_orig = np.isin(labels, ids)
            if mask_orig.sum() == 0:
                continue

            color_img = np.zeros((1, 1, 3), dtype=np.uint8)
            color_img[0, 0] = rgb
            color_lab = cv2.cvtColor(color_img, cv2.COLOR_RGB2LAB).astype(np.float32)
            target_ab = color_lab[0, 0, 1:3]

            # Apply blending inside high-resolution mask
            final_ab[mask_orig] = (blend_weight * target_ab) + ((1.0 - blend_weight) * final_ab[mask_orig])

    final_lab = np.zeros((H_orig, W_orig, 3), dtype=np.float32)
    final_lab[:, :, 0] = L_orig.squeeze(2)
    final_lab[:, :, 1:3] = np.clip(final_ab, 0, 255)

    final_rgb = cv2.cvtColor(final_lab.astype(np.uint8), cv2.COLOR_LAB2RGB)
    colorized_pil = Image.fromarray(final_rgb)

    orig_resized = img_pil.convert("RGB")
    seg_colormap = get_segmentation_colormap(labels)
    seg_resized = Image.fromarray(seg_colormap)

    return colorized_pil, seg_resized, orig_resized, combined_rules

# Gradio Handlers
def detect_regions_handler(img):
    if img is None:
        return "Please upload an image.", None

    labels = segment_image(img)
    total_pixels = labels.size

    detected = []
    for class_name, ids in class_mapping.items():
        mask = np.isin(labels, ids)
        pct = (mask.sum() / total_pixels) * 100.0
        if pct > 0.5:
            detected.append(f"- **{class_name.capitalize()}** ({pct:.1f}% of image area)")

    if not detected:
        region_text = "### Mapped Regions:\n*None detected (>0.5% area).*"
    else:
        region_text = "### Detected Editable Regions:\n" + "\n".join(detected)

    seg_colormap = get_segmentation_colormap(labels)
    seg_pil = Image.fromarray(seg_colormap)

    return region_text, seg_pil

def add_rule_handler(region, color_preset, custom_color, active_rules):
    if active_rules is None:
        active_rules = {}

    presets = {
        "Sky Blue": [135, 206, 235],
        "Forest Green": [34, 139, 34],
        "Grass Green": [50, 205, 50],
        "Red Brick": [178, 34, 34],
        "Concrete Gray": [128, 128, 128],
        "Sunset Orange": [255, 69, 0],
        "Ocean Blue": [30, 144, 255],
        "Yellow Gold": [255, 215, 0],
        "Earth Brown": [139, 69, 19],
        "Snow White": [250, 250, 250]
    }

    if color_preset != "Custom (Use Color Picker Below)":
        rgb = presets.get(color_preset, [128, 128, 128])
    else:
        rgb = hex_to_rgb(custom_color)

    active_rules[region] = rgb

    rule_lines = [f"- **{r.capitalize()}**: RGB {color}" for r, color in active_rules.items()]
    rule_display = "### Active Rules Set:\n" + "\n".join(rule_lines)
    return active_rules, rule_display

def clear_rules_handler():
    return {}, "### Active Rules Set:\n*No manual rules added.*"

def process_colorization(img, text_prompt, blend_weight, active_rules, region_sel, preset_sel, custom_color_sel):
    if img is None:
        return None, None, None, "Please upload an image first.", None

    # If no manual rules are saved in state and no text prompt is entered,
    # automatically apply the currently selected dropdown rule!
    if not active_rules and not text_prompt.strip():
        presets = {
            "Sky Blue": [135, 206, 235],
            "Forest Green": [34, 139, 34],
            "Grass Green": [50, 205, 50],
            "Red Brick": [178, 34, 34],
            "Concrete Gray": [128, 128, 128],
            "Sunset Orange": [255, 69, 0],
            "Ocean Blue": [30, 144, 255],
            "Yellow Gold": [255, 215, 0],
            "Earth Brown": [139, 69, 19],
            "Snow White": [250, 250, 250]
        }
        if preset_sel != "Custom (Use Color Picker Below)":
            rgb = presets.get(preset_sel, [128, 128, 128])
        else:
            rgb = hex_to_rgb(custom_color_sel)
        active_rules = {region_sel: rgb}

    try:
        colorized, seg_map, orig_resized, applied_rules = colorize_image_pipeline(
            img,
            text_prompt=text_prompt,
            dropdown_rules=active_rules,
            blend_weight=blend_weight
        )

        save_path = "outputs/colorized/restored_output.png"
        colorized.save(save_path)

        applied_lines = [f"- **{r.capitalize()}** -> RGB {rgb}" for r, rgb in applied_rules.items()]
        status = "### Colorization Successful!\n**Applied Rules:**\n" + ("\n".join(applied_lines) if applied_lines else "*None*")

        return orig_resized, seg_map, colorized, status, save_path
    except Exception as e:
        import traceback
        return None, None, None, f"Error processing: {str(e)}\n{traceback.format_exc()}", None

# GUI Layout
with gr.Blocks(theme=gr.themes.Soft(primary_hue="teal", secondary_hue="slate")) as demo:
    gr.Markdown("# 🎨 AI-Powered Conditional Image Colorization")
    gr.Markdown("Colorize grayscale images by providing natural language prompts or custom region-based rules.")

    active_rules_state = gr.State({})

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(type="pil", label="1. Upload Grayscale Image", height=300)
            detected_regions_md = gr.Markdown("### Detected Editable Regions:\n*Upload an image to segment.*")

        with gr.Column(scale=2):
            gr.Markdown("### 2. Configure Color Conditions")
            text_prompt_input = gr.Textbox(
                label="Free-Text Prompt (e.g. 'make the sky deep blue, grass bright green, and roads dark gray')",
                placeholder="Enter natural language commands..."
            )

            with gr.Row():
                region_dropdown = gr.Dropdown(
                    choices=["sky", "grass", "vegetation", "water", "building", "road", "sidewalk", "mountain", "cloud", "floor", "ceiling"],
                    value="sky",
                    label="Region Selector"
                )
                preset_dropdown = gr.Dropdown(
                    choices=["Custom (Use Color Picker Below)", "Sky Blue", "Forest Green", "Grass Green", "Red Brick", "Concrete Gray", "Sunset Orange", "Ocean Blue", "Yellow Gold", "Earth Brown", "Snow White"],
                    value="Sky Blue",
                    label="Color Preset"
                )

            custom_color_picker = gr.ColorPicker(value="#4F46E5", label="Custom Color Picker")

            with gr.Row():
                add_rule_btn = gr.Button("Add Rule", variant="secondary")
                clear_rules_btn = gr.Button("Clear Rules", variant="stop")

            active_rules_display = gr.Markdown("### Active Rules Set:\n*No manual rules added.*")

    with gr.Row():
        blend_slider = gr.Slider(minimum=0.0, maximum=1.0, value=0.8, step=0.05, label="Color Bleed Blending Weight")
        colorize_btn = gr.Button("Colorize Grayscale Image", variant="primary")

    gr.Markdown("### 3. Side-by-Side Outputs & Comparisons")
    with gr.Row():
        view_original = gr.Image(label="Original Image (Grayscale)")
        view_segment = gr.Image(label="Segmentation Map (Detected Classes)")
        view_colorized = gr.Image(label="Colorized Output (Conditional)")

    with gr.Row():
        status_output = gr.Markdown("Upload and click Colorize.")
        download_btn = gr.File(label="Download Colorized Output")

    # Actions
    input_image.upload(
        fn=detect_regions_handler,
        inputs=[input_image],
        outputs=[detected_regions_md, view_segment]
    )

    add_rule_btn.click(
        fn=add_rule_handler,
        inputs=[region_dropdown, preset_dropdown, custom_color_picker, active_rules_state],
        outputs=[active_rules_state, active_rules_display]
    )

    clear_rules_btn.click(
        fn=clear_rules_handler,
        outputs=[active_rules_state, active_rules_display]
    )

    colorize_btn.click(
        fn=process_colorization,
        inputs=[input_image, text_prompt_input, blend_slider, active_rules_state, region_dropdown, preset_dropdown, custom_color_picker],
        outputs=[view_original, view_segment, view_colorized, status_output, download_btn]
    )

demo.launch(inline=True, share=True, debug=False)


/tmp/ipykernel_8261/556748508.py:195: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="teal", secondary_hue="slate")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0dcf3aa2c1a07de368.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 7. Performance Evaluation & Metrics
To document the system performance, we calculate standard reconstruction metrics (PSNR, SSIM) and **Region-Specific Color Accuracy (Delta-E)**.


In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric

def evaluate_colorization_sample(img_color_pil, text_prompt="", active_rules=None):
    img_color = img_color_pil.resize((256, 256))
    img_gray = img_color.convert("L")

    colorized, _, _, applied_rules = colorize_image_pipeline(
        img_gray,
        text_prompt=text_prompt,
        dropdown_rules=active_rules,
        blend_weight=1.0
    )

    arr_gt = np.array(img_color)
    arr_pred = np.array(colorized)

    psnr_val = psnr_metric(arr_gt, arr_pred)
    ssim_val = ssim_metric(arr_gt, arr_pred, channel_axis=2)

    print("--- Evaluation Summary ---")
    print(f"PSNR: {psnr_val:.2f} dB")
    print(f"SSIM: {ssim_val:.4f}")

    labels = segment_image(img_gray)
    for region, target_rgb in applied_rules.items():
        if region in class_mapping:
            ids = class_mapping[region]
            mask_orig = np.isin(labels, ids)
            if mask_orig.sum() == 0:
                continue

            mask_256 = cv2.resize(mask_orig.astype(np.uint8), (256, 256), interpolation=cv2.INTER_NEAREST)

            # Average color in L*a*b* space
            pred_lab_all = cv2.cvtColor(arr_pred, cv2.COLOR_RGB2LAB)
            pred_region_lab = pred_lab_all[mask_256 == 1]
            if len(pred_region_lab) == 0:
                continue
            avg_pred_lab = np.mean(pred_region_lab, axis=0)

            color_img = np.zeros((1, 1, 3), dtype=np.uint8)
            color_img[0, 0] = target_rgb
            target_lab = cv2.cvtColor(color_img, cv2.COLOR_RGB2LAB)[0, 0]

            # Delta-E Color Distance
            delta_e = np.sqrt(np.sum((avg_pred_lab - target_lab) ** 2))
            print(f"Region '{region}' Delta-E Color Distance: {delta_e:.2f} (lower is better, <10 is excellent)")

# Run evaluation on one of our dataset images as a demo
demo_img = Image.open(os.path.join("dataset", image_files[0]))
evaluate_colorization_sample(demo_img, text_prompt="make the sky blue and vegetation dark green")
